Question-1

In [120]:
records = [
    {"learner_id": " L01 ", "score": "82.5"},
    {"learner_id": "L02", "score": 60},
    {"learner_id": "L03", "score": None},
    {"learner_id": "   ", "score": 75},
    {"learner_id": "L04", "score": True},
    {"learner_id": "L05", "score": 48.0},
    {"learner_id": "L06", "score": 105},
]
def clean_training_records(records):
    cleaned_records=[]
    rejected=[]
    for record in records:
        clean_id=record["learner_id"].strip()
        raw_score=record["score"]

        if (
            clean_id!="" 
            and raw_score is not None 
            and type(raw_score)!=bool 
            and 0<=float(raw_score)<=100 
            and type(raw_score) in (int, float)):

            raw_score=float(raw_score)
            if raw_score>=60 and raw_score<=100:
                cleaned_records.append({"learner_id":clean_id,"score":raw_score,"eligible":True})
            else:
                cleaned_records.append({"learner_id":clean_id,"score":raw_score,"eligible":False})

        else:
            rejected.append(records.index(record))

    print({"cleaned":cleaned_records,"rejected_indices":rejected})

clean_training_records(records)

clean_training_records([
    {"learner_id": "L01", "score": 50},
    {"learner_id": "L01", "score": 50},
    {"learner_id": "L02", "score": 120},
])

clean_training_records(records)

clean_training_records([
    {"learner_id": "L01", "score": True},
    {"learner_id": "L02", "score": False},
])
clean_training_records(records)

{'cleaned': [{'learner_id': 'L02', 'score': 60.0, 'eligible': True}, {'learner_id': 'L05', 'score': 48.0, 'eligible': False}], 'rejected_indices': [0, 2, 3, 4, 6]}
{'cleaned': [{'learner_id': 'L01', 'score': 50.0, 'eligible': False}, {'learner_id': 'L01', 'score': 50.0, 'eligible': False}], 'rejected_indices': [2]}
{'cleaned': [{'learner_id': 'L02', 'score': 60.0, 'eligible': True}, {'learner_id': 'L05', 'score': 48.0, 'eligible': False}], 'rejected_indices': [0, 2, 3, 4, 6]}
{'cleaned': [], 'rejected_indices': [0, 1]}
{'cleaned': [{'learner_id': 'L02', 'score': 60.0, 'eligible': True}, {'learner_id': 'L05', 'score': 48.0, 'eligible': False}], 'rejected_indices': [0, 2, 3, 4, 6]}


Question-2

In [119]:
payloads = [
    "  AI, helps TEAMS! ",
    b"Build reliable agents.",
    bytearray(b"AI helps."),
    None,
    b"\xff",
]

def prepare_messages(payloads,min_length):
    punctuation=".,!?;:"
    token_list=[]
    indices=[]
    vocab=[]
    for idx,payload in enumerate(payloads):
        cleaned_words=[]
        if type(payload)==str:
            text=payload.strip()
        elif type(payload)==bytearray or type(payload)==bytes:
            try:
                text=payload.decode('utf-8').strip()
            except UnicodeDecodeError:
                indices.append(idx)
                text=""
        else:
            indices.append(idx)
            text=""

        if text:
            text=text.split()
            # print(text)
            for word in text:
                word=word.lower().strip(punctuation)
                # print(word)
                if word and len(word)>=min_length:
                    if word not in vocab:
                        vocab.append(word)
                    cleaned_words.append(word)

            
            # print(cleaned_words)
            if cleaned_words:
                token_list.append(cleaned_words)
            else:
                indices.append(idx)

        else:
            indices.append(idx)

    dict1=dict(token_lists=token_list,vocubulary=sorted(vocab),rejected_indices=indices)
    print(dict1)
    # print(token_list)
    # print(indices)
    # print(sorted(vocab))

prepare_messages(payloads,min_length=3) #default argument

# Test Input
payloads_1 = [
    None,                    # Rejected (Index 0)
    "Valid message",          # Valid
    None,                    # Rejected again! (Index 2)
    b"Short",                # Valid
    None                     # Rejected third time! (Index 4)
]

# Run the test
prepare_messages(payloads_1, min_length=3)

# Test Input
payloads_2 = [
    "!!!AI!!! ...rules...",   # Extreme punctuation boundary
    "Is it ok? No!",          # "Is", "it", "ok", "No" are all < 3 chars after stripping
    "System-wide update!!",   # Contains a hyphen (not in your punctuation list)
    "   ",                    # Whitespace only
]

# Run the test
prepare_messages(payloads_2, min_length=3)



{'token_lists': [['helps', 'teams'], ['build', 'reliable', 'agents'], ['helps']], 'vocubulary': ['agents', 'build', 'helps', 'reliable', 'teams'], 'rejected_indices': [3, 3, 4, 4]}
{'token_lists': [['valid', 'message'], ['short']], 'vocubulary': ['message', 'short', 'valid'], 'rejected_indices': [0, 0, 2, 2, 4, 4]}
{'token_lists': [['rules'], ['system-wide', 'update']], 'vocubulary': ['rules', 'system-wide', 'update'], 'rejected_indices': [1, 3]}


Question-3

In [ ]:
results = [
    {"id": "T1", "actual": "Billing", "predicted": " billing "},
    {"id": "T2", "actual": "Technical", "predicted": "billing"},
    {"id": "T3", "actual": "billing", "predicted": "billing"},
    {"id": "T4", "actual": "Account", "predicted": "ACCOUNT"},
    {"id": "T5", "actual": None, "predicted": "account"},
]

def summarize_predictions(results):
    # id, actual, predicted should not be None or empty
    # trim surrounding whitespaces and lowercase the two labels

    valid_count = 0
    count = 0
    rejected = []
    grouping = []
    miss_classified = []

    for idx, record in enumerate(results):

        if not all(key in record for key in ["id", "actual", "predicted"]):
            rejected.append(idx)
            continue

        if record["id"] is not None:
            record["id"] = record["id"].strip()

        if record["actual"] is not None:
            record["actual"] = record["actual"].strip().lower()

        if record["predicted"] is not None:
            record["predicted"] = record["predicted"].strip().lower()

        var = True

        if record["actual"] not in grouping and record["actual"] is not None:
            grouping.append(record["actual"])

        for val in [record["id"], record["actual"], record["predicted"]]:
            if val is None or str(val).strip() == "":
                if idx not in rejected:
                    rejected.append(idx)
                var = False

        if var:
            valid_count += 1

            if record["actual"] == record["predicted"]:
                count += 1
            else:
                miss_classified.append(record["id"])

    grouping_arr = {}

    for item in grouping:
        grouping_arr.update({
            str(item): {
                "total": 0,
                "correct": 0
            }
        })


    for record in results:
        actual_key = str(record["actual"])

        if (
            record["actual"] in grouping
            and record["id"] is not None
            and str(record["id"]).strip() != ""
            and record["predicted"] is not None
            and str(record["predicted"]).strip() != ""
        ):
            grouping_arr[actual_key]["total"] += 1

            if record["actual"] == record["predicted"]:
                grouping_arr[actual_key]["correct"] += 1

    final_dict = {
        "valid_count": valid_count,
        "correct_count": count,
        "by_actual": grouping_arr,
        "missclassified_ids": miss_classified,
        "rejected_indices": rejected
    }

    print(final_dict)


summarize_predictions(results)
results = [
    {"id": "T1", "actual": "Billing", "predicted": " billing "},
    {"id": "T2", "actual": "Technical", "predicted": "billing"},
    {"id": "T3", "actual": "billing", "predicted": "billing"},
    {"id": "T4", "actual": "Account", "predicted": "ACCOUNT"},
    {"id": "T5", "actual": None, "predicted": "account"},
]

def summarize_predictions(results):
    # id, actual, predicted should not be None or empty
    # trim surrounding whitespaces and lowercase the two labels

    valid_count = 0
    count = 0
    rejected = []
    grouping = []
    miss_classified = []

    for idx, record in enumerate(results):

        if not all(key in record for key in ["id", "actual", "predicted"]):
            rejected.append(idx)
            continue

        if record["id"] is not None:
            record["id"] = record["id"].strip()

        if record["actual"] is not None:
            record["actual"] = record["actual"].strip().lower()

        if record["predicted"] is not None:
            record["predicted"] = record["predicted"].strip().lower()

        var = True

        if record["actual"] not in grouping and record["actual"] is not None:
            grouping.append(record["actual"])

        for val in [record["id"], record["actual"], record["predicted"]]:
            if val is None or str(val).strip() == "":
                if idx not in rejected:
                    rejected.append(idx)
                var = False


        if var:
            valid_count += 1

            if record["actual"] == record["predicted"]:
                count += 1
            else:
                miss_classified.append(record["id"])

    grouping_arr = {}

    for item in grouping:
        grouping_arr.update({
            str(item): {
                "total": 0,
                "correct": 0
            }
        })

  
    for record in results:
        actual_key = str(record["actual"])

        if (
            record["actual"] in grouping
            and record["id"] is not None
            and str(record["id"]).strip() != ""
            and record["predicted"] is not None
            and str(record["predicted"]).strip() != ""
        ):
            grouping_arr[actual_key]["total"] += 1

            if record["actual"] == record["predicted"]:
                grouping_arr[actual_key]["correct"] += 1

    final_dict = {
        "valid_count": valid_count,
        "correct_count": count,
        "by_actual": grouping_arr,
        "missclassified_ids": miss_classified,
        "rejected_indices": rejected
    }

    print(final_dict)


summarize_predictions(results)
results = [
    {"id": "T1", "actual": "Billing", "predicted": " billing "},
    {"id": "T2", "actual": "Technical", "predicted": "billing"},
    {"id": "T3", "actual": "billing", "predicted": "billing"},
    {"id": "T4", "actual": "Account", "predicted": "ACCOUNT"},
    {"id": "T5", "actual": None, "predicted": "account"},
]

def summarize_predictions(results):
    # id, actual, predicted should not be None or empty
    # trim surrounding whitespaces and lowercase the two labels

    valid_count = 0
    count = 0
    rejected = []
    grouping = []
    miss_classified = []

    for idx, record in enumerate(results):


        if not all(key in record for key in ["id", "actual", "predicted"]):
            rejected.append(idx)
            continue

        if record["id"] is not None:
            record["id"] = record["id"].strip()

        if record["actual"] is not None:
            record["actual"] = record["actual"].strip().lower()

        if record["predicted"] is not None:
            record["predicted"] = record["predicted"].strip().lower()

        var = True

        if record["actual"] not in grouping and record["actual"] is not None:
            grouping.append(record["actual"])

        for val in [record["id"], record["actual"], record["predicted"]]:
            if val is None or str(val).strip() == "":
                if idx not in rejected:
                    rejected.append(idx)
                var = False

        # CHANGE 2: only valid records are counted
        if var:
            valid_count += 1

            if record["actual"] == record["predicted"]:
                count += 1
            else:
                miss_classified.append(record["id"])

    grouping_arr = {}

    for item in grouping:
        grouping_arr.update({
            str(item): {
                "total": 0,
                "correct": 0
            }
        })

    for record in results:
        actual_key = str(record["actual"])

        if (
            record["actual"] in grouping
            and record["id"] is not None
            and str(record["id"]).strip() != ""
            and record["predicted"] is not None
            and str(record["predicted"]).strip() != ""
        ):
            grouping_arr[actual_key]["total"] += 1

            if record["actual"] == record["predicted"]:
                grouping_arr[actual_key]["correct"] += 1

    final_dict = {
        "valid_count": valid_count,
        "correct_count": count,
        "by_actual": grouping_arr,
        "missclassified_ids": miss_classified,
        "rejected_indices": rejected
    }

    print(final_dict)


summarize_predictions(results)
results = [
    {"id": "T1", "actual": "Billing", "predicted": " billing "},
    {"id": "T2", "actual": "Technical", "predicted": "billing"},
    {"id": "T3", "actual": "billing", "predicted": "billing"},
    {"id": "T4", "actual": "Account", "predicted": "ACCOUNT"},
    {"id": "T5", "actual": None, "predicted": "account"},
]

def summarize_predictions(results):
    # id, actual, predicted should not be None or empty
    # trim surrounding whitespaces and lowercase the two labels

    valid_count = 0
    count = 0
    rejected = []
    grouping = []
    miss_classified = []

    for idx, record in enumerate(results):

        if not all(key in record for key in ["id", "actual", "predicted"]):
            rejected.append(idx)
            continue

        if record["id"] is not None:
            record["id"] = record["id"].strip()

        if record["actual"] is not None:
            record["actual"] = record["actual"].strip().lower()

        if record["predicted"] is not None:
            record["predicted"] = record["predicted"].strip().lower()

        var = True

        if record["actual"] not in grouping and record["actual"] is not None:
            grouping.append(record["actual"])

        for val in [record["id"], record["actual"], record["predicted"]]:
            if val is None or str(val).strip() == "":
                if idx not in rejected:
                    rejected.append(idx)
                var = False

        # CHANGE 2: only valid records are counted
        if var:
            valid_count += 1

            if record["actual"] == record["predicted"]:
                count += 1
            else:
                miss_classified.append(record["id"])

    grouping_arr = {}

    for item in grouping:
        grouping_arr.update({
            str(item): {
                "total": 0,
                "correct": 0
            }
        })

    for record in results:
        actual_key = str(record["actual"])

        if (
            record["actual"] in grouping
            and record["id"] is not None
            and str(record["id"]).strip() != ""
            and record["predicted"] is not None
            and str(record["predicted"]).strip() != ""
        ):
            grouping_arr[actual_key]["total"] += 1

            if record["actual"] == record["predicted"]:
                grouping_arr[actual_key]["correct"] += 1

    final_dict = {
        "valid_count": valid_count,
        "correct_count": count,
        "by_actual": grouping_arr,
        "missclassified_ids": miss_classified,
        "rejected_indices": rejected
    }

    print(final_dict)


summarize_predictions(results)

{'valid_count': 4, 'correct_count': 3, 'by_actual': {'billing': {'total': 2, 'correct': 2}, 'technical': {'total': 1, 'correct': 0}, 'account': {'total': 1, 'correct': 1}}, 'missclassified_ids': ['T2'], 'rejected_indices': [4]}
{'valid_count': 4, 'correct_count': 3, 'by_actual': {'billing': {'total': 2, 'correct': 2}, 'technical': {'total': 1, 'correct': 0}, 'account': {'total': 1, 'correct': 1}}, 'missclassified_ids': ['T2'], 'rejected_indices': [4]}
{'valid_count': 4, 'correct_count': 3, 'by_actual': {'billing': {'total': 2, 'correct': 2}, 'technical': {'total': 1, 'correct': 0}, 'account': {'total': 1, 'correct': 1}}, 'missclassified_ids': ['T2'], 'rejected_indices': [4]}
{'valid_count': 4, 'correct_count': 3, 'by_actual': {'billing': {'total': 2, 'correct': 2}, 'technical': {'total': 1, 'correct': 0}, 'account': {'total': 1, 'correct': 1}}, 'missclassified_ids': ['T2'], 'rejected_indices': [4]}


Question-4

In [115]:
train_ids = ["R3", "R1", " R2 ", "R1", None]
validation_ids = ["R2", "R4", "R4", " "]
from collections import Counter

def audit_dataset_ids(train_ids,validation_ids):
    #accept non empty strings after trimming
    #keep identifier case-sensitive
    #empty list produce empty frozenset
    #order may differ in frozenset but its content should match
    #snapshot must remain unchanged even ist original list is modified later
    stripped_train_ids=[item.strip() for item in train_ids if type(item)==str and item.strip()!=""]
    stripped_validation_ids=[item.strip() for item in validation_ids if type(item)==str and item.strip()!=""]
    invalid_train_ids = [
    idx for idx, item in enumerate(train_ids)
    if type(item) != str or (type(item) == str and item.strip() == "")
    ]
    invalid_validation_ids = [
    idx for idx, item in enumerate(validation_ids)
    if type(item) != str or (type(item) == str and item.strip() == "")
    ]
    set_train_only = set(stripped_train_ids)
    set_validation_only = set(stripped_validation_ids)
    train_id_count=Counter(stripped_train_ids)
    trained_duplicates=[i for i,count in train_id_count.items() if count>1]
    validation_id_count=Counter(stripped_validation_ids)
    validation_duplicates=[i for i,count in validation_id_count.items() if count>1]
    overlap_ids = sorted(set_train_only & set_validation_only)
    train_only = sorted(set_train_only - set_validation_only)
    validation_only = sorted(set_validation_only - set_train_only)
    frozen_train_id=frozenset(stripped_train_ids)
    frozen_validation_id=frozenset(i for i in stripped_validation_ids if i.strip()!='')
    

    # print(train_ids)
    # print(stripped_train_ids)
    # print(validation_ids)
    # print(stripped_validation_ids)
    # print(train_only)
    # print(set_train_only)
    # print(set_validation_only)
    # print(validation_only)
    # print(trained_duplicates)
    # print(validation_duplicates)
    # print(overlap_ids)
    # print(invalid_train_ids)
    # print(invalid_validation_ids)
    # print(frozen_train_id)
    # print(frozen_validation_id)

    dict1=dict(trained_duplicates=trained_duplicates,validation_duplicates=validation_duplicates,overlap=overlap_ids,train_only=train_only,validation_only=validation_only,invalid_train_indices=invalid_train_ids,invalid_validation_indices=invalid_validation_ids,snapshots=dict(train=frozen_train_id,validation=frozen_validation_id))
    print(dict1)
audit_dataset_ids(train_ids, validation_ids)
audit_dataset_ids([], [])
audit_dataset_ids(
    [" R1 ", "   ", None, "R2"],
    [" R1 ", "R3", " "]
)


{'trained_duplicates': ['R1'], 'validation_duplicates': ['R4'], 'overlap': ['R2'], 'train_only': ['R1', 'R3'], 'validation_only': ['R4'], 'invalid_train_indices': [4], 'invalid_validation_indices': [3], 'snapshots': {'train': frozenset({'R3', 'R2', 'R1'}), 'validation': frozenset({'R4', 'R2'})}}
{'trained_duplicates': [], 'validation_duplicates': [], 'overlap': [], 'train_only': [], 'validation_only': [], 'invalid_train_indices': [], 'invalid_validation_indices': [], 'snapshots': {'train': frozenset(), 'validation': frozenset()}}
{'trained_duplicates': [], 'validation_duplicates': [], 'overlap': ['R1'], 'train_only': ['R2'], 'validation_only': ['R3'], 'invalid_train_indices': [1, 2], 'invalid_validation_indices': [2], 'snapshots': {'train': frozenset({'R1', 'R2'}), 'validation': frozenset({'R3', 'R1'})}}


Question_5

In [110]:
queue = ["D3", "D1", "D3", "D2"]
incoming = ["D4", "D2", "D5"]
completed = ["D1", "D9","D3","D2","D4","D5"]




def update_annotation_queue(queue,incoming,completed,priority,batch_size):
    combined_queue=[]
    if batch_size<=0:
        raise ValueError
    for i in queue:
        if i not in combined_queue:
            combined_queue.append(i)
    for i in incoming:
        if i not in combined_queue:
            combined_queue.append(i)
    
    for i in completed:
        if i in combined_queue:
            combined_queue.remove(i)
    if priority is not None:
        if priority in combined_queue:
            combined_queue.remove(priority)
        combined_queue.insert(0, priority)
    batched_queue=combined_queue[:batch_size]
    last_element=combined_queue[-1] if combined_queue else None
    position=[]
    for id,item in enumerate(combined_queue,1):
        position.append((id,item))
    print(last_element)
    print(combined_queue)
    print(batched_queue)
    print(position)


    print("Result")
    dict5=dict(queue=combined_queue,next_batch=batched_queue,last_item=last_element,positions=position)
    print(dict5)

update_annotation_queue(
    queue, incoming, completed,
    priority=None, batch_size=8,
)

queue = ["D3", "D1", "D3", "D2"]
a=queue
print(a)
print(queue)
a.append("d10")
print(a)
print(queue)
print(a is queue)
b=queue.copy()
b.append("d11")
print(b)
print(queue)
print(b is queue)

update_annotation_queue(
    ["D1", "D2"],
    ["D3"],
    ["D1", "D2", "D3"],
    priority=None,
    batch_size=5
)

update_annotation_queue(
    ["D1", "D2", "D3"],
    [],
    [],
    priority="D2",
    batch_size=2
)

None
[]
[]
[]
Result
{'queue': [], 'next_batch': [], 'last_item': None, 'positions': []}
['D3', 'D1', 'D3', 'D2']
['D3', 'D1', 'D3', 'D2']
['D3', 'D1', 'D3', 'D2', 'd10']
['D3', 'D1', 'D3', 'D2', 'd10']
True
['D3', 'D1', 'D3', 'D2', 'd10', 'd11']
['D3', 'D1', 'D3', 'D2', 'd10']
False
None
[]
[]
[]
Result
{'queue': [], 'next_batch': [], 'last_item': None, 'positions': []}
D3
['D2', 'D1', 'D3']
['D2', 'D1']
[(1, 'D2'), (2, 'D1'), (3, 'D3')]
Result
{'queue': ['D2', 'D1', 'D3'], 'next_batch': ['D2', 'D1'], 'last_item': 'D3', 'positions': [(1, 'D2'), (2, 'D1'), (3, 'D3')]}


Interview follow up: Why would assigning the original queue to another variable fail to create an independent queue?

Answer: On assigning original queue to new variable will
 point the new variable to the same address location of the original queue, so any change in queue with new variable will directly affect the main queue. It can be examplified by the above example where "a is queue" which return True. Now to create independent queue we need to use copy() or deepcopy() which will create new queue and point to new memory location. Any amendment in this queue will not affect the original queue. It can be examplified by the example of "b is queue" which returns False since it points to new memory location.

Question_6

In [107]:
import copy
base = {
    "model": "ticket_classifier",
    "parameters": {"threshold": 0.5, "max_tokens": 128},
    "metadata": {"owner": "platform", "stage": "baseline"},
}

def build_experiment_config(base, * args, run_name,parameter_updates, **metadata):
    a=copy.deepcopy(base)
    if parameter_updates is None:
        parameter_updates={}
    if type(parameter_updates)==dict:
        a["parameters"]={**a["parameters"],**parameter_updates}
    else:
        raise TypeError

    a["metadata"]={**a["metadata"],**metadata}

    if type(run_name)!=str:
        raise TypeError
    if run_name.strip()=="":
        raise ValueError
    run_name=run_name.strip()
        
    a["run_name"]=run_name

    print(a)
    print(base)

    # dict6=dict(model=a["model"],parameters=a["parameters"],metadata=a["metadata"],run_name=a["run_name"])
    # print(dict6)
    # print("Base",base)
    # print("Args",args)
    # print("Run_name",run_name)
    # print("Parameter_updates",parameter_updates)
    # print("Metadata",metadata)
try:
    build_experiment_config(
        base,
        run_name="trial_04",
        parameter_updates=["threshold", 0.7]
    )
except TypeError:
    print("TypeError caught")
build_experiment_config(
    base,
    run_name="trial_03",
    parameter_updates=None
)
# build_experiment_config(
#     base,
#     run_name="   ",
#     parameter_updates={"threshold": 0.7}
# )

build_experiment_config(
    base, run_name="trial_02",
    parameter_updates={"threshold": 0.7},
    owner="nlp_team", seed=42,
)


TypeError caught
{'model': 'ticket_classifier', 'parameters': {'threshold': 0.5, 'max_tokens': 128}, 'metadata': {'owner': 'platform', 'stage': 'baseline'}, 'run_name': 'trial_03'}
{'model': 'ticket_classifier', 'parameters': {'threshold': 0.5, 'max_tokens': 128}, 'metadata': {'owner': 'platform', 'stage': 'baseline'}}
{'model': 'ticket_classifier', 'parameters': {'threshold': 0.7, 'max_tokens': 128}, 'metadata': {'owner': 'nlp_team', 'stage': 'baseline', 'seed': 42}, 'run_name': 'trial_02'}
{'model': 'ticket_classifier', 'parameters': {'threshold': 0.5, 'max_tokens': 128}, 'metadata': {'owner': 'platform', 'stage': 'baseline'}}


Interview follow up: Under what circumstances would a shallow dictionary copy be insufficient?

Answer: A shallow copy is insufficient whenever a dictionary contains nested mutable objects such as list,tuple,dictionaries,sets or custom object instance and you intend to modify these items independently.

Question_7

In [99]:
batch_a = [80, 90, None]
batch_b = (90, 70, True)
batch_c = []


def aggregate_scores(*batches,top_n):
    proper_item_list=[]
    count=0
    rejected_indices=[]
    top_score_list=[]
    # print(batches)
    if top_n<0:
        raise ValueError
    
    for id,item in enumerate(batches):
        if type(item)!=list and type(item)!=tuple:
            raise TypeError
        
        
        for id_inside,i in enumerate(item):
            if type(i)!=bool and (type(i)==float or type(i)==int):
                if 0<=i<=100:
                    count+=1
                    proper_item_list.append(i)
                    top_score_list.append((id,id_inside,i))
                else:
                    rejected_indices.append((id, id_inside))
            else:
                rejected_indices.append((id,id_inside))
    if len(proper_item_list)==0:
        mean=0
    else:
        mean=round(sum(proper_item_list)/len(proper_item_list),2)
    # print(proper_item_list)
    # print(sorted_set)
    top_score_list=sorted(top_score_list,reverse=True,key=lambda x:x[2])
    
    # print(top_score_list)
    # print(mean)
    # print(count)
    # print(rejected_indices)

    dict_7=dict(valid_count=count,mean=mean,top_score=top_score_list[:top_n],rejected_position=rejected_indices)
    print(dict_7)
aggregate_scores([], top_n=3)
aggregate_scores([None, True, "90"], top_n=3)
aggregate_scores([80, -1, 101], top_n=3)
aggregate_scores(batch_a, batch_b, batch_c, top_n=3)

{'valid_count': 0, 'mean': 0, 'top_score': [], 'rejected_position': []}
{'valid_count': 0, 'mean': 0, 'top_score': [], 'rejected_position': [(0, 0), (0, 1), (0, 2)]}
{'valid_count': 1, 'mean': 80.0, 'top_score': [(0, 0, 80)], 'rejected_position': [(0, 1), (0, 2)]}
{'valid_count': 4, 'mean': 82.5, 'top_score': [(0, 1, 90), (1, 0, 90), (0, 0, 80)], 'rejected_position': [(0, 2), (1, 2)]}


In [94]:
batch_a = [80, 90, None]
batch_b = (90, 70, True)
batch_c = []


def aggregate_scores(*batches,top_n):
    proper_item_list=[]
    count=0
    rejected_indices=[]
    top_score_list=[]
    # print(batches)
    if top_n<0:
        raise ValueError
    
    for id,item in enumerate(batches):
        if type(item)!=list and type(item)!=tuple:
            raise TypeError
        
        
        for id_inside,i in enumerate(item):
            if (type(i)!=bool and type(i)==float) or (type(i)==int):
                if i>=0 and i<=100:
                    count+=1
                    proper_item_list.append(i)
                    top_score_list.append((id,id_inside,i))
            else:
                rejected_indices.append((id,id_inside)) 
    if len(proper_item_list)==0:
        mean=0
    else:
        mean=round(sum(proper_item_list)/len(proper_item_list),2)
    # print(proper_item_list)
    # print(sorted_set)
    top_score_list=sorted(top_score_list,reverse=True,key=lambda x:x[2])
    
    # print(top_score_list)
    # print(mean)
    # print(count)
    # print(rejected_indices)

    dict_7=dict(valid_count=count,mean=mean,top_score=top_score_list[:top_n],rejected_position=rejected_indices)
    print(dict_7)
# aggregate_scores([80, 90], top_n=0)
# aggregate_scores([], top_n=3)
aggregate_scores(batch_a, batch_b, batch_c, top_n=3)

{'valid_count': 4, 'mean': 82.5, 'top_score': [(0, 1, 90), (1, 0, 90), (0, 0, 80)], 'rejected_position': [(0, 2), (1, 2)]}


Interview follow up: When can a generator expression reduce memory usage compared with a list comprehension?

Answer: A generator expression reduces memory usage compared with a list comprehension when you are processing data that can be evaluated one element at a time (lazily), rather than needing the entire dataset available in memory simultaneously (eagerly).

Question_8

In [87]:

def summarise_tool_calls(record):
    final_records={item["call_id"]: item for item in record}
    rejected_indices=[]
    by_tool={}
    for item in final_records.values():
        tool_name = item["tool"]
        status = item["status"]
        duration = item["duration_ms"]
        rejected_indices=[item["call_id"] for item in final_records.values() if item["status"]=="error"]
        if tool_name not in by_tool:
            by_tool[tool_name] = {
                "calls": 0,
                "successes": 0,
                "failures": 0,
                "total_duration_ms": 0
            }
        by_tool[tool_name]["calls"] += 1
        by_tool[tool_name]["total_duration_ms"] += duration

        if status == "ok":
            by_tool[tool_name]["successes"] += 1
        elif status == "error":
            by_tool[tool_name]["failures"] += 1

    tool_ranking = sorted(by_tool, key=lambda x: by_tool[x]["failures"], reverse=True)

    output = {
        "unique_calls": len(final_records),
        "by_tool": by_tool,
        "failed_call_ids": sorted(rejected_indices), 
        "tool_ranking": tool_ranking
    }

    # print(count)
    # print(rejected_indices)
    # print(final_records)
    # unique_tool=sorted(unique_tool,reverse=True)
    # print(by_tool)
    print(output)
    
    
records = [
    {"call_id": "C1", "tool": "search",
     "status": "error", "duration_ms": 120},
    {"call_id": "C2", "tool": "calculator",
     "status": "ok", "duration_ms": 10},
    {"call_id": "C1", "tool": "search",
     "status": "ok", "duration_ms": 80},
    {"call_id": "C3", "tool": "search",
     "status": "error", "duration_ms": 40},
    {"call_id": "C4", "tool": "retriever",
     "status": "error", "duration_ms": 50},
]

records = [
    {"call_id": "C1", "tool": "search", "status": "ok", "duration_ms": 80},
    {"call_id": "C1", "tool": "search", "status": "error", "duration_ms": 120},
]
summarise_tool_calls([])


summarise_tool_calls(records)

{'unique_calls': 0, 'by_tool': {}, 'failed_call_ids': [], 'tool_ranking': []}
{'unique_calls': 1, 'by_tool': {'search': {'calls': 1, 'successes': 0, 'failures': 1, 'total_duration_ms': 120}}, 'failed_call_ids': ['C1'], 'tool_ranking': ['search']}


Interview follow up: How does the last-record-wins requirement affect your choice of data structure?

Answer: The "last-record-wins" requirement directly dictates using a standard Python dictionary (or hash map) as the primary data structure for the deduplication step.

Question_9


In [82]:
from functools import wraps

texts = ["  agent ", "ML", "data"]

def validate_text_batch(func):

        @wraps(func)
        def wrapper(*args, **kwargs):
            if args:
                 texts=args[0]
            elif "texts" in kwargs:
                 texts=kwargs["texts"]
            if type(texts)!=list:
                raise TypeError
            for item in texts:
                if type(item)!=str:
                    raise TypeError
                
            for item in texts:
                if item.strip()=="":
                     raise ValueError
            return func(*args, **kwargs)
        return wrapper

@validate_text_batch
def text_length(texts,*,strip=True):
    length=[]

    if strip:
        length=[len(item.strip()) for item in texts]
    else:
        length=[len(item) for item in texts]

    return length



a=text_length(texts)
b=text_length(texts=texts,strip=False)
c=text_length([])
try:
    d = text_length(["hello", "   "])
except ValueError:
    d = "ValueError"
try:
    e=text_length(["hello", 123])
except TypeError:
    e="TypeError"
print(a)
print(b)
print(c)
print(d)
print(e)

[5, 2, 4]
[8, 2, 4]
[]
ValueError
TypeError


Interview follow up: Why is preserving a decorated function’s name and docstring useful in a larger codebase?

Answer: The decorator changes how a function behaves, but @wraps helps prevent it from changing the function's identity and documentation.

Question_10

In [ ]:
from functools import wraps
import math

audit_logs=[]
call_id = 0

def decorator(func):
    @wraps(func)
    def wrapper(*args,**kwargs):
        call_id=len(audit_logs)+1
        
        try:
            result=func(*args,**kwargs)
            # print("Result",result)
            # call_id+=1
            audit_logs.append(dict(call_number=call_id,function=func.__name__,status="success",error_type=None))
            return result
        except Exception as e:
            # call_id+=1
            audit_logs.append(dict(call_number=call_id,function=func.__name__,status="error",error_type=e.__class__.__name__))
            raise

        return func(*args,*kwargs)
    return wrapper

@decorator
def average_score(l):
    if type(l)!=list:
        raise TypeError

    for item in l:
        if type(item)==bool:
            raise TypeError
        if type(item)!=int and type(item)!=float:
            raise TypeError
        if not math.isfinite(item):
            raise ValueError

    if len(l)>0:
        a=sum(item for item in l)
        return a/len(l)
    else:
        raise ValueError
        

@decorator   
def format_run_name(eval,*,number):
    if type(eval)!=str:
        raise TypeError
    if eval=="":
        raise ValueError
    if type(number)==bool:
        raise TypeError
    if type(number)!=int:
        raise TypeError
    if number<0:
        raise ValueError
    return f"{eval}_{number}"

a=average_score([10,20,30,40])
b=format_run_name("eval",number=3)
# print(a,b)

try:
    c = average_score([])
except ValueError:
    pass
# d=average_score([10, True, 20])
# e=average_score([10, float("inf")])
print(audit_logs)

TypeError: 

Interview follow up:What problems could arise if every decorated function stored its records in one global history list?

Answer: A single global history list makes all decorated functions share the same records and call numbers, even when they belong to separate tasks or evaluation runs. It also makes testing harder because one test or run can affect another. Separate history lists allow each group of functions to maintain its own independent audit trail.